Below I’ll give you modular PyTorch functions for:

- Real Möbius map (M_w(x)) on ball / sphere  
- Kuramoto-on-sphere order parameter and vector field  
- Reduced hyperbolic (w)-dynamics (both via Möbius and via autograd gradient)  
- Simple simulation + Plotly visualization hooks for 2D (disk/S¹) and 3D (ball/S²)  

Everything is real-valued, dot-product based, and autograd-friendly.

---

## 1. Core math building blocks (PyTorch)


In [1]:
import torch
from torch import Tensor
import math


# 1.1 Basic helpers
def normalize(x: Tensor, eps: float = 1e-9) -> Tensor:
    """
    Normalize vectors along the last dimension.
    x: [..., d]
    returns: [..., d] with unit norm (or zero if input ~0).
    """
    norm = x.norm(dim=-1, keepdim=True)
    return x / (norm + eps)


def dot(a: Tensor, b: Tensor) -> Tensor:
    """
    Dot product along the last dimension.
    a, b: [..., d]
    returns: [...] (scalar per batch entry)
    """
    return (a * b).sum(dim=-1)


### 1.2 Real Möbius map on ball / sphere

This is the real version of the Möbius map \(M_w\) we discussed:

\[
M_w(x)
= \frac{(1 - |w|^2)(x - |x|^2 w)}{1 - 2\langle w,x\rangle + |w|^2 |x|^2} - w.
\]


In [2]:
def mobius_ball(x: Tensor, w: Tensor, eps: float = 1e-9) -> Tensor:
    """
    Real Möbius map M_w(x) on the unit ball B^d.

    x: [..., d]   points in R^d, ideally |x| < 1
    w: [..., d] or [d]  parameter vector in B^d (|w| < 1)
       (broadcastable to shape of x)

    returns: [..., d]  = M_w(x)
    """
    # ensure broadcasting
    while w.dim() < x.dim():
        w = w.unsqueeze(0)

    x2 = (x * x).sum(dim=-1, keepdim=True)         # |x|^2
    w2 = (w * w).sum(dim=-1, keepdim=True)         # |w|^2
    wx = (w * x).sum(dim=-1, keepdim=True)         # <w, x>

    num = (1.0 - w2) * (x - x2 * w)
    den = 1.0 - 2.0 * wx + w2 * x2
    den = den.clamp(min=eps)

    return num / den - w


def mobius_sphere(x: Tensor, w: Tensor, eps: float = 1e-9) -> Tensor:
    """
    Möbius map restricted to points on the unit sphere S^{d-1}.
    Uses the simplified formula valid when |x|=1:

    M_w(x) = ((1 - |w|^2)(x - w))/|x - w|^2 - w
    """
    while w.dim() < x.dim():
        w = w.unsqueeze(0)

    w2 = (w * w).sum(dim=-1, keepdim=True)         # |w|^2
    diff = x - w
    diff2 = (diff * diff).sum(dim=-1, keepdim=True)  # |x - w|^2
    diff2 = diff2.clamp(min=eps)

    return (1.0 - w2) * diff / diff2 - w


### 1.3 Hyperbolic metric factor & potential

Hyperbolic metric on the ball \(B^d\):

\[
g_{\text{hyp}} = \phi(w)^2 I, \quad
\phi(w) = \frac{2}{1 - |w|^2}.
\]

Relation between Euclidean and hyperbolic gradients:

\[
\nabla_{\text{hyp}}\Phi(w) = \frac{(1 - |w|^2)^2}{4}\,\nabla\Phi(w).
\]

Hyperbolic potential for linear order parameter:

\[
\Phi(w) = \sum_i a_i \log\frac{|w - p_i|^2}{1 - |w|^2}.
\]


In [3]:
def hyperbolic_conformal_factor(w: Tensor, eps: float = 1e-9) -> Tensor:
    """
    phi(w) = 2 / (1 - |w|^2)
    returns: scalar or [...,1]
    """
    w2 = (w * w).sum(dim=-1, keepdim=True)
    return 2.0 / ((1.0 - w2).clamp(min=eps))


def hyperbolic_potential(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor,
    eps: float = 1e-9
) -> Tensor:
    """
    Phi(w) = sum_i a_i log( |w - p_i|^2 / (1 - |w|^2) )

    w: [d] or [...,d]
    base_points: [N,d]
    weights: [N]
    returns: scalar potential (broadcasted if w has leading dims)
    """
    # make w shape [..., 1, d] to broadcast with [N,d]
    orig_shape = w.shape[:-1]
    d = w.shape[-1]
    w_expanded = w.view(*orig_shape, 1, d)         # [..., 1, d]

    diff = w_expanded - base_points               # [..., N, d]
    num = (diff * diff).sum(dim=-1)               # [..., N]
    num = num.clamp(min=eps)

    w2 = (w * w).sum(dim=-1, keepdim=False)       # [...]
    denom = (1.0 - w2).clamp(min=eps)             # [...]

    # log( |w - p_i|^2 / (1 - |w|^2) )
    log_term = torch.log(num / denom.unsqueeze(-1))   # [..., N]

    # weights [N] -> broadcast to [..., N]
    phi = (log_term * weights.unsqueeze(0)).sum(dim=-1)  # [...]

    return phi


### 1.4 Hyperbolic gradient via autograd

We’ll let autograd handle the Euclidean gradient and then convert to hyperbolic gradient.


In [4]:
def hyperbolic_grad_autograd(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor,
    create_graph: bool = False
) -> Tensor:
    """
    Compute hyperbolic gradient ∇_hyp Phi(w) using autograd.

    w: [d] with requires_grad=True
    base_points: [N,d]
    weights: [N]

    returns: [d]  = ∇_hyp Phi(w)
    """
    assert w.requires_grad, "w must have requires_grad=True"

    phi = hyperbolic_potential(w, base_points, weights)  # scalar
    (grad_euc,) = torch.autograd.grad(
        phi, w, create_graph=create_graph, retain_graph=create_graph
    )

    w2 = (w * w).sum()
    factor = ((1.0 - w2).clamp(min=1e-9) ** 2) / 4.0

    grad_hyp = factor * grad_euc
    return grad_hyp


## 2. Kuramoto / alignment vector fields in real form

### 2.1 Order parameter in transformed coordinates

Given a base configuration \((p_i \in S^{d-1})\), and parameter \(w(t)\in B^d\), the current directions are

\[
x_i(t) = M_w(p_i),
\]

if we ignore the extra rotation \(\zeta_t\) for now.  
Then the linear order parameter is

\[
Z(w) = \sum_i a_i x_i(w) = \sum_i a_i M_w(p_i).
\]


In [5]:
def mobius_pushforward_points(
    w: Tensor, base_points: Tensor
) -> Tensor:
    """
    x_i(w) = M_w(p_i) on the sphere (|p_i|=1).

    w: [d]
    base_points: [N,d]
    returns: [N,d]
    """
    return mobius_sphere(base_points, w)


def order_parameter(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor
) -> Tensor:
    """
    Z(w) = sum_i a_i M_w(p_i)

    w: [d]
    base_points: [N,d]
    weights: [N]
    returns: [d]
    """
    x = mobius_pushforward_points(w, base_points)     # [N,d]
    return (weights[:, None] * x).sum(dim=0)          # [d]


### 2.2 Reduced w-dynamics (WS-style, real)

From the L–M–S reduction (with identical A and linear Z), we can use the simple form

\[
\dot w = -\frac12(1 - |w|^2)\,Z(w), \quad
Z(w) = \sum_i a_i M_w(p_i).
\]

Alternatively, you can drive \(w\) directly by the negative hyperbolic gradient of \(\Phi\).


In [6]:
def w_vector_field_mobius(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor
) -> Tensor:
    """
    Reduced WS-type vector field for w in B^d:

    dw/dt = -0.5 * (1 - |w|^2) * Z(w),
    with Z(w) = sum_i a_i M_w(p_i).

    w: [d]
    base_points: [N,d]
    weights: [N]

    returns: [d] (dw/dt)
    """
    Z = order_parameter(w, base_points, weights)   # [d]
    w2 = (w * w).sum()
    factor = -0.5 * (1.0 - w2)
    return factor * Z


def w_vector_field_grad(
    w: Tensor,
    base_points: Tensor,
    weights: Tensor
) -> Tensor:
    """
    Hyperbolic gradient flow:

    dw/dt = -∇_hyp Phi(w)

    (Up to sign conventions; this uses autograd.)
    """
    w = w.clone().requires_grad_(True)
    grad_hyp = hyperbolic_grad_autograd(w, base_points, weights)
    return -grad_hyp.detach()


You can use either `w_vector_field_mobius` (explicit WS formula) or `w_vector_field_grad` (concept-check via autograd) and compare them numerically.

### 2.3 Direct Kuramoto / alignment on the sphere

If you also want the “full” real Kuramoto for directions (without reduction):

\[
\dot x_i = A x_i + Z - \langle Z,x_i\rangle x_i.
\]


In [7]:
def kuramoto_sphere_vector_field(
    x: Tensor,
    A: Tensor,
    weights: Tensor
) -> Tensor:
    """
    Kuramoto / alignment vector field on S^{d-1}:

    x: [N,d]  (assumed |x_i|=1)
    A: [d,d]  skew-symmetric intrinsic rotation
    weights: [N]  alignment weights a_i

    returns: [N,d] = dx/dt
    """
    # x_i in current configuration
    Z = (weights[:, None] * x).sum(dim=0)          # [d]
    proj = (x * Z)                                 # elementwise
    proj = proj.sum(dim=-1, keepdim=True) * x      # <Z,x_i> x_i
    Ax = x @ A.T                                   # [N,d]
    return Ax + Z - proj


## 3. Simple simulation + Plotly visualization hooks

Here’s a tiny Euler integrator for the reduced \(w(t)\) dynamics, plus examples for 2D and 3D visualization.

### 3.1 Simple Euler integration for \(w(t)\)


In [8]:
def integrate_w_euler(
    w0: Tensor,
    base_points: Tensor,
    weights: Tensor,
    dt: float,
    steps: int,
    field="mobius"  # or "grad"
):
    """
    Simple Euler integration of w dynamics.

    returns: [steps+1, d] trajectory
    """
    traj = []
    w = w0.clone()
    for _ in range(steps + 1):
        traj.append(w.detach().cpu().clone())
        if field == "mobius":
            dw = w_vector_field_mobius(w, base_points, weights)
        else:
            dw = w_vector_field_grad(w, base_points, weights)
        w = w + dt * dw
        # keep inside ball (to be safe numerically)
        norm = w.norm()
        if norm >= 0.999:
            w = w / (norm + 1e-9) * 0.999
    return torch.stack(traj, dim=0)   # [T,d]


### 3.2 2D case: Kuramoto on S¹ via real embedding

For \(d=2\), you can embed Kuramoto phases \(\theta_k\) as unit vectors:

\[
x_k = (\cos\theta_k,\ \sin\theta_k) \in S^1.
\]

We’ll also set up a simple example configuration and integrate the reduced dynamics.


In [9]:
def phases_to_s1_torch(theta: Tensor) -> Tensor:
    """
    theta: [N] phases
    returns: [N,2] points on S^1
    """
    return torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)


# Example: N points on S^1 as base configuration p_i
N = 8
theta0 = torch.linspace(0, 2*math.pi, N+1)[:-1]
p2 = phases_to_s1_torch(theta0)       # [N,2]
weights2 = torch.ones(N)

# start w at origin
w0_2d = torch.zeros(2)

traj_w_2d = integrate_w_euler(
    w0_2d, p2, weights2, dt=0.05, steps=200
)  # [T,2]

# reconstruct directions at some time t*
t_star = -1
x_tstar = mobius_pushforward_points(traj_w_2d[t_star], p2)  # [N,2]


In [10]:
def random_points_on_sphere(N: int, d: int = 3) -> Tensor:
    """
    Sample N random unit vectors in R^d.
    """
    x = torch.randn(N, d)
    return normalize(x)


# Example: base configuration p_i on S^2
N = 64
p3 = random_points_on_sphere(N, d=3)   # [N,3]
weights3 = torch.ones(N)

w0_3d = torch.zeros(3)                 # start at hyperbolic origin
traj_w_3d = integrate_w_euler(w0_3d, p3, weights3, dt=0.001, steps=200)


You can now plot:

- `traj_w_2d` as a path in the unit disk,  
- `x_tstar` as points on the unit circle (e.g. with Plotly Scatter).


You could similarly make a `plot_disk_2d(traj_w_2d)` to watch the hyperbolic parameter \(w(t)\) move around inside the unit disk.

---

## How this all fits your goal

**Modularity**:

- Every mathematical object (Möbius map, order parameter, potential, hyperbolic gradient, Kuramoto vector field) is its own standalone function on torch tensors.

**Real-valued / matrix-friendly**:

- Everything is written with dot products, broadcasting, and real tensors – no complex numbers.

**Autograd-ready**:

- `hyperbolic_potential` is a pure tensor function.  
- `hyperbolic_grad_autograd` uses `torch.autograd.grad` to get \(\nabla_{\text{hyp}}\Phi\), which you can reuse to test or optimize parameters.

**Visualizations**:

- 2D: use `traj_w_2d` in the disk + `mobius_pushforward_points` to see how base phases on the circle move.  
- 3D: use `traj_w_3d` + Plotly sphere plotting to see the evolving configuration on \(S^2\).

If you’d like, next step could be:

- wrapping this into a small `nn.Module` that learns weights \(a_i\) to fit observed velocity-direction data,  
- or adding intrinsic rotations \(A\) and visualizing the combined effect in these hyperbolic coordinates.


### 3.3 3D case: Kuramoto / alignment on S²

For \(d=3\), you can choose any base configuration on \(S^2\). For example, random points:


In [11]:
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# --- Assume your mobius_sphere is already defined and dimension-agnostic ---
# def mobius_sphere(x: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
#     ...

def _sphere_wireframe_traces(n_lat: int = 9, n_lon: int = 18):
    """
    Build lat-long wireframe curves on S^2 as Scatter3d traces.
    Returns: list[go.Scatter3d]
    """
    traces = []
    # Latitudes (horizontal circles)
    lat_vals = np.linspace(-0.8 * np.pi / 2, 0.8 * np.pi / 2, n_lat)
    lon = np.linspace(0, 2*np.pi, 200)
    for phi in lat_vals:
        x = np.cos(phi) * np.cos(lon)
        y = np.cos(phi) * np.sin(lon)
        z = np.sin(phi) * np.ones_like(lon)
        traces.append(go.Scatter3d(
            x=x.tolist(), y=y.tolist(), z=z.tolist(),
            mode="lines",
            line=dict(color="lightgrey", width=1),
            hoverinfo="skip",
            showlegend=False
        ))
    # Longitudes (vertical circles)
    lon_vals = np.linspace(0, 2*np.pi, n_lon, endpoint=False)
    lat = np.linspace(-np.pi/2, np.pi/2, 200)
    for lam in lon_vals:
        x = (np.cos(lat) * np.cos(lam))
        y = (np.cos(lat) * np.sin(lam))
        z = np.sin(lat)
        traces.append(go.Scatter3d(
            x=x.tolist(), y=y.tolist(), z=z.tolist(),
            mode="lines",
            line=dict(color="lightgrey", width=1),
            hoverinfo="skip",
            showlegend=False
        ))
    return traces


class Kuramoto3DWidget:
    """
    Interactive Dashboard for 3D Kuramoto / Möbius Dynamics on S^2.

    Layout:
    [ Left Col: S^2 Visualization ]  [ Right Col: Time Slider & Metric Plots ]
    """

    def __init__(
        self,
        traj_w: torch.Tensor,       # [T, 3] trajectory in B^3
        base_points: torch.Tensor,  # [N, 3] points on S^2
        title: str = "3D Kuramoto Dynamics on S²",
        point_size: int = 5,
        w_color: str = "red",
        osc_color: str = "blue",
        show_w_inside_ball: bool = True,
    ):
        # --- Data ---
        self.traj_w = traj_w.detach().cpu()
        self.base_points = base_points.detach().cpu()
        self.T = self.traj_w.shape[0]
        self.point_size = point_size
        self.w_color = w_color
        self.osc_color = osc_color
        self.show_w_inside_ball = show_w_inside_ball

        # 1. Precompute x_i(t) and metrics
        self._precompute_dynamics()

        # 2. Init figures
        self._init_sphere_figure(title)
        self._init_metrics_figure()

        # 3. Init widgets
        self._init_widgets()

        # Layout
        self.layout = widgets.HBox([
            self.sphere_fig,
            widgets.VBox([
                self.controls_box,
                self.metrics_fig
            ])
        ])
        display(self.layout)

    # -------------------------------------------------------------
    # Precomputation
    # -------------------------------------------------------------
    def _precompute_dynamics(self):
        x_all = []
        metric_w_mag = []
        metric_order = []

        for t in range(self.T):
            w_t = self.traj_w[t]              # [3]
            x_t = mobius_sphere(self.base_points, w_t)  # [N,3]
            x_t = x_t / (x_t.norm(dim=-1, keepdim=True) + 1e-9)
            x_all.append(x_t)

            w_mag = float(torch.norm(w_t))
            metric_w_mag.append(w_mag)

            centroid = torch.mean(x_t, dim=0)
            r_mag = float(torch.norm(centroid))
            metric_order.append(r_mag)

        self.x_all = torch.stack(x_all).numpy()   # [T,N,3]
        self.traj_w_np = self.traj_w.numpy()      # [T,3]

        self.metrics_data = {
            "w_mag": np.array(metric_w_mag),
            "order_p": np.array(metric_order),
        }
        self.time_steps = np.arange(self.T)

    # -------------------------------------------------------------
    # Sphere figure
    # -------------------------------------------------------------
    def _init_sphere_figure(self, title):
        self.sphere_fig = go.FigureWidget()

        # Static wireframe
        wire_traces = _sphere_wireframe_traces()
        for tr in wire_traces:
            self.sphere_fig.add_trace(tr)

        self._wire_count = len(self.sphere_fig.data)

        # Initial dynamic traces
        x0 = self.x_all[0]         # [N,3]
        w0 = self.traj_w_np[0]     # [3]
        path0 = self.traj_w_np[:1] # [1,3]

        # Oscillators
        self.sphere_fig.add_trace(go.Scatter3d(
            x=x0[:, 0].tolist(),
            y=x0[:, 1].tolist(),
            z=x0[:, 2].tolist(),
            mode="markers",
            marker=dict(size=self.point_size, color=self.osc_color),
            name="Oscillators"
        ))

        # w(t)
        self.sphere_fig.add_trace(go.Scatter3d(
            x=[float(w0[0])],
            y=[float(w0[1])],
            z=[float(w0[2])],
            mode="markers",
            marker=dict(size=self.point_size+2, color=self.w_color, symbol="x"),
            name="w(t)",
            visible=self.show_w_inside_ball
        ))

        # w path
        self.sphere_fig.add_trace(go.Scatter3d(
            x=path0[:, 0].tolist(),
            y=path0[:, 1].tolist(),
            z=path0[:, 2].tolist(),
            mode="lines",
            line=dict(color=self.w_color, width=2, dash="dot"),
            name="w path",
            hoverinfo="skip",
            visible=self.show_w_inside_ball
        ))

        self.sphere_fig.update_layout(
            title=title,
            width=650, height=650,
            scene=dict(
                aspectmode="data",
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
            ),
            margin=dict(l=20, r=20, t=50, b=20),
            showlegend=False
        )

    # -------------------------------------------------------------
    # Metrics figure
    # -------------------------------------------------------------
    def _init_metrics_figure(self):
        self.metrics_fig = go.FigureWidget(make_subplots(
            rows=2, cols=1,
            subplot_titles=("Order Parameter R(t)", "Control Magnitude |w(t)|"),
            vertical_spacing=0.15
        ))

        # R(t)
        self.metrics_fig.add_trace(go.Scatter(
            x=self.time_steps.tolist(),
            y=self.metrics_data["order_p"].tolist(),
            mode="lines",
            line=dict(color='purple'),
            name="R(t)"
        ), row=1, col=1)

        self.metrics_fig.add_trace(go.Scatter(
            x=[0],
            y=[float(self.metrics_data["order_p"][0])],
            mode='markers',
            marker=dict(size=10, color='red'),
            showlegend=False
        ), row=1, col=1)

        # |w(t)|
        self.metrics_fig.add_trace(go.Scatter(
            x=self.time_steps.tolist(),
            y=self.metrics_data["w_mag"].tolist(),
            mode="lines",
            line=dict(color='green'),
            name="|w(t)|"
        ), row=2, col=1)

        self.metrics_fig.add_trace(go.Scatter(
            x=[0],
            y=[float(self.metrics_data["w_mag"][0])],
            mode='markers',
            marker=dict(size=10, color='red'),
            showlegend=False
        ), row=2, col=1)

        self.metrics_fig.update_layout(
            width=600, height=500,
            margin=dict(l=20, r=20, t=40, b=20),
            showlegend=False
        )

        self.metrics_fig.update_xaxes(title_text="Time", range=[0, self.T], row=1, col=1)
        self.metrics_fig.update_xaxes(title_text="Time", range=[0, self.T], row=2, col=1)
        self.metrics_fig.update_yaxes(range=[-0.05, 1.05], row=1, col=1)
        self.metrics_fig.update_yaxes(range=[-0.05, 1.05], row=2, col=1)

    # -------------------------------------------------------------
    # Widgets
    # -------------------------------------------------------------
    def _init_widgets(self):
        self.play = widgets.Play(
            value=0, min=0, max=self.T-1,
            step=1, interval=50,
            description="Press play",
            show_repeat=False
        )

        self.slider = widgets.IntSlider(
            value=0, min=0, max=self.T-1,
            description="Time:",
            continuous_update=False,  # small extra robustness
            layout=widgets.Layout(width='300px')
        )

        widgets.jslink((self.play, 'value'), (self.slider, 'value'))
        self.slider.observe(self._update_frame, names='value')

        self.controls_box = widgets.HBox([self.play, self.slider])

    # -------------------------------------------------------------
    # Callback
    # -------------------------------------------------------------
    def _update_frame(self, change):
        t = int(change['new'])

        x_t = self.x_all[t]             # [N,3]
        w_t = self.traj_w_np[t]         # [3]
        path_t = self.traj_w_np[:t+1]   # [<=T,3]

        idx_osc  = self._wire_count
        idx_w    = self._wire_count + 1
        idx_path = self._wire_count + 2

        # 1) Update sphere fig
        with self.sphere_fig.batch_update():
            self.sphere_fig.data[idx_osc].x = x_t[:, 0].tolist()
            self.sphere_fig.data[idx_osc].y = x_t[:, 1].tolist()
            self.sphere_fig.data[idx_osc].z = x_t[:, 2].tolist()

            self.sphere_fig.data[idx_w].x = [float(w_t[0])]
            self.sphere_fig.data[idx_w].y = [float(w_t[1])]
            self.sphere_fig.data[idx_w].z = [float(w_t[2])]

            self.sphere_fig.data[idx_path].x = path_t[:, 0].tolist()
            self.sphere_fig.data[idx_path].y = path_t[:, 1].tolist()
            self.sphere_fig.data[idx_path].z = path_t[:, 2].tolist()

        # 2) Update metrics fig dots
        with self.metrics_fig.batch_update():
            # R(t) dot → data[1]
            self.metrics_fig.data[1].x = [t]
            self.metrics_fig.data[1].y = [float(self.metrics_data["order_p"][t])]

            # |w| dot → data[3]
            self.metrics_fig.data[3].x = [t]
            self.metrics_fig.data[3].y = [float(self.metrics_data["w_mag"][t])]

In [12]:
# traj_w_3d: [T,3] in B^3
# p3: [N,3] on S^2

widget3d = Kuramoto3DWidget(traj_w_3d, p3)

    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', …

In [13]:
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# --- 1. Helper Functions (Assumed defined, but included for standalone runnable) ---

def mobius_sphere(x: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    """
    Applies the Möbius transformation defined by w to points x.
    Map: f_w(x) = (1 - |w|^2) / |x - w|^2 * (x - w) - w
    (Standard form for B^n vector Möbius transformation)
    """
    # x: [N, D], w: [D] or [1, D] broadcasted
    w = w.view(1, -1)
    diff = x - w
    # Squared norms
    w_sq = (w**2).sum(dim=1, keepdim=True)
    diff_sq = (diff**2).sum(dim=1, keepdim=True)
    
    # The formula M_w(x) = (1-|w|^2)/|x-w|^2 * (x-w) - w
    # Note: This is the inversion-based form often used in geometric DL
    scale = (1 - w_sq) / (diff_sq + 1e-8)
    return scale * diff - w

# --- 2. The Interactive Widget ---

class KuramotoWidget:
    """
    Interactive Dashboard for 2D Kuramoto/Möbius Dynamics.
    
    Layout:
    [ Left Col: Disk Animation ]  [ Right Col: Time Slider & Metric Plots ]
    """
    def __init__(
        self,
        traj_w: torch.Tensor,
        base_points: torch.Tensor,
        title: str = "2D Kuramoto Dynamics",
        point_size: int = 8,
        w_color: str = "red",
        osc_color: str = "blue",
    ):
        # --- Data Prep ---
        self.traj_w = traj_w.detach().cpu()
        self.base_points = base_points.detach().cpu()
        self.T = self.traj_w.shape[0]
        self.point_size = point_size
        self.w_color = w_color
        self.osc_color = osc_color
        
        # 1. Precompute Simulation State (Positions & Metrics)
        self._precompute_dynamics()
        
        # 2. Build Figures
        self._init_disk_figure(title)
        self._init_metrics_figure()
        
        # 3. Build Widgets & Layout
        self._init_widgets()
        
        # 4. Display
        # Layout: HBox( [DiskFigure, VBox([Controls, MetricsFigure])] )
        self.layout = widgets.HBox([
            self.disk_fig,
            widgets.VBox([
                self.controls_box, 
                self.metrics_fig
            ])
        ])
        display(self.layout)

    def _precompute_dynamics(self):
        """Pre-calculates all x_i(t) and metric values for the slider."""
        x_all = []
        metric_w_mag = []
        metric_order = []
        
        # We perform the loop once here to keep the slider callback fast
        for t in range(self.T):
            w_t = self.traj_w[t]
            
            # Dynamics: x(t) = M_{w(t)}(base)
            x_t = mobius_sphere(self.base_points, w_t)
            x_all.append(x_t)
            
            # Metric 1: Magnitude of w
            w_mag = torch.norm(w_t).item()
            metric_w_mag.append(w_mag)
            
            # Metric 2: Order Parameter R = | (1/N) * sum(x_i) |
            # (Centroid magnitude)
            centroid = torch.mean(x_t, dim=0)
            r_mag = torch.norm(centroid).item()
            metric_order.append(r_mag)
            
        self.x_all = torch.stack(x_all).numpy() # [T, N, 2]
        self.traj_w_np = self.traj_w.numpy()    # [T, 2]
        
        # Metrics Data
        self.metrics_data = {
            "w_mag": np.array(metric_w_mag),
            "order_p": np.array(metric_order)
        }
        self.time_steps = np.arange(self.T)

    def _init_disk_figure(self, title):
        """Initializes the Left Panel (Poincaré Disk)."""
        self.disk_fig = go.FigureWidget()
        
        # Static: Unit Circle
        theta = np.linspace(0, 2*np.pi, 200)
        self.disk_fig.add_trace(go.Scatter(
            x=np.cos(theta), y=np.sin(theta), mode='lines', 
            line=dict(color='black', width=2), hoverinfo='skip', name="Boundary"
        ))
        
        # Dynamic: Oscillators (Trace 1)
        x0 = self.x_all[0]
        self.disk_fig.add_trace(go.Scatter(
            x=x0[:,0], y=x0[:,1], mode='markers',
            marker=dict(size=self.point_size, color=self.osc_color),
            name="Oscillators"
        ))
        
        # Dynamic: w(t) (Trace 2)
        w0 = self.traj_w_np[0]
        self.disk_fig.add_trace(go.Scatter(
            x=[w0[0]], y=[w0[1]], mode='markers',
            marker=dict(size=self.point_size+4, color=self.w_color, symbol='x'),
            name="w(t)"
        ))
        
        # Dynamic: w path (Trace 3)
        self.disk_fig.add_trace(go.Scatter(
            x=[w0[0]], y=[w0[1]], mode='lines',
            line=dict(color=self.w_color, width=1, dash='dot'),
            name="w path"
        ))

        self.disk_fig.update_layout(
            title=title, width=600, height=600,
            xaxis=dict(range=[-1.1, 1.1], visible=False, scaleanchor="y", scaleratio=1),
            yaxis=dict(range=[-1.1, 1.1], visible=False),
            margin=dict(l=20, r=20, t=50, b=20),
            showlegend=False
        )

    def _init_metrics_figure(self):
        """Initializes the Right Panel (Time Series Metrics)."""
        self.metrics_fig = go.FigureWidget(make_subplots(
            rows=2, cols=1, 
            subplot_titles=("Order Parameter R(t)", "Control Magnitude |w(t)|"),
            vertical_spacing=0.15
        ))
        
        # --- Plot 1: Order Parameter ---
        # Full history line
        self.metrics_fig.add_trace(go.Scatter(
            x=self.time_steps, y=self.metrics_data["order_p"],
            mode='lines', line=dict(color='purple'), name="R(t)"
        ), row=1, col=1)
        # Current time dot
        self.metrics_fig.add_trace(go.Scatter(
            x=[0], y=[self.metrics_data["order_p"][0]],
            mode='markers', marker=dict(size=10, color='red'), showlegend=False
        ), row=1, col=1)
        
        # --- Plot 2: |w| Magnitude ---
        # Full history line
        self.metrics_fig.add_trace(go.Scatter(
            x=self.time_steps, y=self.metrics_data["w_mag"],
            mode='lines', line=dict(color='green'), name="|w(t)|"
        ), row=2, col=1)
        # Current time dot
        self.metrics_fig.add_trace(go.Scatter(
            x=[0], y=[self.metrics_data["w_mag"][0]],
            mode='markers', marker=dict(size=10, color='red'), showlegend=False
        ), row=2, col=1)

        self.metrics_fig.update_layout(
            width=500, height=500,
            margin=dict(l=20, r=20, t=40, b=20),
            showlegend=False
        )
                
        # --- FIX: Explicitly lock X-Axis ranges so they don't move ---
        self.metrics_fig.update_xaxes(title_text="Time", range=[0, self.T], row=1, col=1)
        self.metrics_fig.update_xaxes(title_text="Time", range=[0, self.T], row=2, col=1)
        
        # Optional: Lock Y-axes to reasonable defaults (0 to 1) if desired
        self.metrics_fig.update_yaxes(range=[-0.05, 1.05], row=1, col=1) # R is [0,1]
        self.metrics_fig.update_yaxes(range=[-0.05, 1.05], row=2, col=1) # |w| is < 1

    def _init_widgets(self):
        """Sets up the Play, Pause, and Slider controls."""
        self.play = widgets.Play(
            value=0, min=0, max=self.T-1,
            step=1, interval=50, 
            description="Press play",
            show_repeat=False
        )
        
        self.slider = widgets.IntSlider(
            value=0, min=0, max=self.T-1,
            description="Time:",
            layout=widgets.Layout(width='300px')
        )
        
        widgets.jslink((self.play, 'value'), (self.slider, 'value'))
        self.slider.observe(self._update_frame, names='value')
        
        self.controls_box = widgets.HBox([self.play, self.slider])

    def _update_frame(self, change):
        """Callback: Updates both figures when slider moves."""
        t = change['new']
        
        # 1. Update Disk
        x_t = self.x_all[t]
        w_t = self.traj_w_np[t]
        path_slice = self.traj_w_np[:t+1]
        
        with self.disk_fig.batch_update():
            # Oscillators
            self.disk_fig.data[1].x = x_t[:, 0]
            self.disk_fig.data[1].y = x_t[:, 1]
            # w(t) marker
            self.disk_fig.data[2].x = [w_t[0]]
            self.disk_fig.data[2].y = [w_t[1]]
            # w path
            self.disk_fig.data[3].x = path_slice[:, 0]
            self.disk_fig.data[3].y = path_slice[:, 1]

        # 2. Update Metrics (Only move the red dot)
        with self.metrics_fig.batch_update():
            # R(t) dot (Trace 1)
            self.metrics_fig.data[1].x = [t]
            self.metrics_fig.data[1].y = [self.metrics_data["order_p"][t]]
            
            # |w| dot (Trace 3)
            self.metrics_fig.data[3].x = [t]
            self.metrics_fig.data[3].y = [self.metrics_data["w_mag"][t]]

In [14]:


# 2. Instantiate and display
widget = KuramotoWidget(traj_w_2d, p2)

    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'black', 'wid…

# Fibered

In [15]:
import torch

def fiber_full_cross_entropy_vector_field(
    ws: torch.Tensor,           # [K, dim] current WS/L-M-S variables w_k(t)
    fiber_points: torch.Tensor, # [K, N, dim] u_k^i on S^{dim-1}
    dim: int,
) -> torch.Tensor:
    """
    For each fiber k, compute dw_k/dt as the negative hyperbolic gradient
    of the full within–cross potential:

      Φ_k(w_k)
        = (dim-1) Σ_i log P(w_k, u_k^i)
          - Σ_{ℓ≠k} Σ_j log P(w_k, -u_ℓ^j)

    implemented via hyperbolic_potential(w_k, base_pts, weights).

    ws:          [K, dim]
    fiber_points:[K, N, dim] (unit vectors on S^{dim-1})
    dim:         ambient dimension (e.g. 3 for S^2)
    returns:     [K, dim]   dw_k/dt for each fiber
    """
    K, N, d = fiber_points.shape
    assert d == dim, "dim must match fiber_points.shape[-1]"
    device = ws.device

    d_minus_1 = dim 
    dw_all = []

    for k in range(K):
        # w_k is the WS/L-M-S variable for fiber k; it evolves via full objective
        w_k = ws[k].detach().clone().requires_grad_(True)

        # Internal points: u_k^i with weight (dim-1)
        self_pts = fiber_points[k]                           # [N, d]
        self_w  = torch.full((N,), float(d_minus_1), device=device)

        # External anti-points: -u_ℓ^j for ℓ≠k with weight -1
        if K > 1:
            others = torch.cat(
                [fiber_points[ell] for ell in range(K) if ell != k],
                dim=0
            )                                                # [(K-1)*N, d]
            ext_pts = -others
            ext_w   = -torch.ones(ext_pts.shape[0], device=device)
            base_pts = torch.cat([self_pts, ext_pts], dim=0) # [N + (K-1)N, d]
            weights  = torch.cat([self_w, ext_w], dim=0)     # [N + (K-1)N]
        else:
            base_pts = self_pts
            weights  = self_w

        # Hyperbolic gradient ∇_hyp Φ_k(w_k)
        grad_hyp = hyperbolic_grad_autograd(
            w_k, base_pts, weights, create_graph=False
        )  # [d]

        # Natural gradient flow: dw_k/dt = -∇_hyp Φ_k(w_k)
        dw_k = -grad_hyp.detach()
        dw_all.append(dw_k)

    return torch.stack(dw_all, dim=0)  # [K, dim]

In [16]:
def integrate_fiber_full_cross_entropy_euler(
    w0_all: torch.Tensor,       # [K, dim] initial WS variables w_k(0)
    fiber_points: torch.Tensor, # [K, N, dim] u_k^i on S^{dim-1}
    dt: float,
    steps: int,
    dim: int,
) -> torch.Tensor:
    """
    Euler integration of K fibers' WS variables under the full
    cross-anti-alignment / within-alignment hyperbolic objective:

      dw_k/dt = -∇_hyp Φ_k(w_k)

    w0_all is just the initial condition (e.g. origin, small random, etc.).
    """
    ws = w0_all.clone()
    traj = []

    for _ in range(steps + 1):
        traj.append(ws.detach().cpu().clone())   # [K, dim]

        # full cross-entropy vector field for all fibers
        dw_all = fiber_full_cross_entropy_vector_field(
            ws, fiber_points, dim
        )  # [K, dim]

        ws = ws + dt * dw_all

        # keep each w_k inside the open ball B^dim
        norms = ws.norm(dim=-1, keepdim=True)              # [K, 1]
        too_big = norms >= 0.999                           # [K, 1] bool

        # scale rows with norm ≥ 0.999 down to norm 0.999
        scaled_ws = ws / (norms + 1e-9) * 0.999           # [K, dim]

        # torch.where broadcasts condition [K,1] over last dim
        ws = torch.where(too_big, scaled_ws, ws)          # [K, dim]

    return torch.stack(traj, dim=0)   # [T, K, dim], T = steps+1

In [17]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

# Assumes _sphere_wireframe_traces and mobius_sphere are defined

class FiberCrossEntropyKuramoto3DWidget:
    """
    Interactive 3D visualization for K fibered hyperbolic Kuramoto-style dynamics.

    - traj_w_all: [T, K, 3] trajectory of K fibers' w_k(t) in B^3
    - fiber_points: [K, N, 3] initial fiber points u_k^i on S^2
    """

    def __init__(
        self,
        traj_w_all: torch.Tensor,
        fiber_points: torch.Tensor,
        title: str = "Fibered Hyperbolic Alignment on S²",
        point_size: int = 5,
        fiber_colors: list = None,
    ):
        self.traj_w_all = traj_w_all.detach().cpu()     # [T,K,3]
        self.fiber_points = fiber_points.detach().cpu() # [K,N,3]
        self.T, self.K, _ = self.traj_w_all.shape
        self.N = self.fiber_points.shape[1]

        self.point_size = point_size

        # color palette for fibers
        if fiber_colors is None:
            palette = ["red", "blue", "green", "orange", "purple",
                       "brown", "magenta", "cyan"]
            self.fiber_colors = (palette * ((self.K // len(palette)) + 1))[:self.K]
        else:
            self.fiber_colors = fiber_colors

        # 1) Precompute x_{k,i}(t) and metrics
        self._precompute_dynamics()

        # 2) Initialize sphere + metrics figures
        self._init_sphere_figure(title)
        self._init_metrics_figure()

        # 3) Widgets
        self._init_widgets()

        self.layout = widgets.HBox([
            self.sphere_fig,
            widgets.VBox([
                self.controls_box,
                self.metrics_fig
            ])
        ])
        display(self.layout)

    # ---------------------------------------------------------
    # Precomputation: x_{k,i}(t) and metrics
    # ---------------------------------------------------------
    def _precompute_dynamics(self):
        x_all = []      # [T, K, N, 3]
        w_mag_all = []  # [T, K]
        fiber_order = []# [T, K]
        global_order = [] # [T]

        for t in range(self.T):
            ws_t = self.traj_w_all[t]      # [K,3]
            x_k_list = []
            w_mag_t = []
            order_k_t = []

            # per-fiber Möbius pushforward
            for k in range(self.K):
                w_k = ws_t[k]                              # [3]
                pts_k = self.fiber_points[k]              # [N,3]
                x_k = mobius_sphere(pts_k, w_k)           # [N,3]
                x_k = x_k / (x_k.norm(dim=-1, keepdim=True) + 1e-9)
                x_k_list.append(x_k)

                w_mag_t.append(float(torch.norm(w_k)))
                centroid_k = x_k.mean(dim=0)
                order_k_t.append(float(torch.norm(centroid_k)))

            x_k_stack = torch.stack(x_k_list, dim=0)     # [K,N,3]
            x_all.append(x_k_stack)

            w_mag_all.append(w_mag_t)
            fiber_order.append(order_k_t)

            # global order parameter (all fibers combined)
            all_pts = x_k_stack.view(-1, 3)              # [K*N,3]
            centroid_global = all_pts.mean(dim=0)
            global_order.append(float(torch.norm(centroid_global)))

        self.x_all = torch.stack(x_all, dim=0).numpy()         # [T,K,N,3]
        self.traj_w_np = self.traj_w_all.numpy()               # [T,K,3]
        self.w_mag_all = np.array(w_mag_all)                   # [T,K]
        self.fiber_order = np.array(fiber_order)               # [T,K]
        self.global_order = np.array(global_order)             # [T]
        self.time_steps = np.arange(self.T)

    # ---------------------------------------------------------
    # Sphere figure
    # ---------------------------------------------------------
    def _init_sphere_figure(self, title):
        self.sphere_fig = go.FigureWidget()
        wire_traces = _sphere_wireframe_traces()
        for tr in wire_traces:
            self.sphere_fig.add_trace(tr)
        self._wire_count = len(self.sphere_fig.data)

        # initial dynamic data
        x0 = self.x_all[0]              # [K,N,3]
        ws0 = self.traj_w_np[0]         # [K,3]
        paths0 = self.traj_w_np[:1]     # [1,K,3]

        # 1) fiber point clouds (one trace per fiber)
        self._points_start_idx = self._wire_count
        for k in range(self.K):
            xk0 = x0[k]                 # [N,3]
            color = self.fiber_colors[k]
            self.sphere_fig.add_trace(go.Scatter3d(
                x=xk0[:, 0].tolist(),
                y=xk0[:, 1].tolist(),
                z=xk0[:, 2].tolist(),
                mode="markers",
                marker=dict(size=self.point_size, color=color),
                name=f"fiber {k} points"
            ))

        # 2) w_k markers
        self._w_start_idx = self._points_start_idx + self.K
        for k in range(self.K):
            w0 = ws0[k]
            color = self.fiber_colors[k]
            self.sphere_fig.add_trace(go.Scatter3d(
                x=[float(w0[0])],
                y=[float(w0[1])],
                z=[float(w0[2])],
                mode="markers",
                marker=dict(size=self.point_size+2, color=color, symbol="x"),
                name=f"w_{k}(t)",
                visible=True
            ))

        # 3) w_k paths
        self._path_start_idx = self._w_start_idx + self.K
        for k in range(self.K):
            path0 = paths0[:, k, :]  # [1,3]
            color = self.fiber_colors[k]
            self.sphere_fig.add_trace(go.Scatter3d(
                x=path0[:, 0].tolist(),
                y=path0[:, 1].tolist(),
                z=path0[:, 2].tolist(),
                mode="lines",
                line=dict(color=color, width=2, dash="dot"),
                name=f"w_{k} path",
                hoverinfo="skip",
                visible=True
            ))

        self.sphere_fig.update_layout(
            title=title,
            width=650, height=650,
            scene=dict(
                aspectmode="data",
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
            ),
            margin=dict(l=20, r=20, t=50, b=20),
            showlegend=False
        )

    # ---------------------------------------------------------
    # Metrics figure
    # ---------------------------------------------------------
    def _init_metrics_figure(self):
        self.metrics_fig = go.FigureWidget(make_subplots(
            rows=2, cols=1,
            subplot_titles=("Global Order Parameter R(t)", "max_k R_k(t) & max_k |w_k(t)|"),
            vertical_spacing=0.15
        ))

        # Global R(t)
        self.metrics_fig.add_trace(go.Scatter(
            x=self.time_steps.tolist(),
            y=self.global_order.tolist(),
            mode="lines",
            line=dict(color='purple'),
            name="R_global(t)"
        ), row=1, col=1)

        self.metrics_fig.add_trace(go.Scatter(
            x=[0],
            y=[float(self.global_order[0])],
            mode='markers',
            marker=dict(size=10, color='red'),
            showlegend=False
        ), row=1, col=1)

        # max_k R_k(t) and max_k |w_k(t)|
        max_Rk = self.fiber_order.max(axis=1)  # [T]
        max_wk = self.w_mag_all.max(axis=1)    # [T]

        self.metrics_fig.add_trace(go.Scatter(
            x=self.time_steps.tolist(),
            y=max_Rk.tolist(),
            mode="lines",
            line=dict(color='blue'),
            name="max_k R_k(t)"
        ), row=2, col=1)

        self.metrics_fig.add_trace(go.Scatter(
            x=self.time_steps.tolist(),
            y=max_wk.tolist(),
            mode="lines",
            line=dict(color='green'),
            name="max_k |w_k(t)|"
        ), row=2, col=1)

        # moving dots
        self.metrics_fig.add_trace(go.Scatter(
            x=[0], y=[float(max_Rk[0])],
            mode='markers',
            marker=dict(size=10, color='red'),
            showlegend=False
        ), row=2, col=1)
        self.metrics_fig.add_trace(go.Scatter(
            x=[0], y=[float(max_wk[0])],
            mode='markers',
            marker=dict(size=10, color='orange'),
            showlegend=False
        ), row=2, col=1)

        self.metrics_fig.update_layout(
            width=600, height=500,
            margin=dict(l=20, r=20, t=40, b=20),
            showlegend=False
        )
        self.metrics_fig.update_xaxes(title_text="Time", range=[0, self.T], row=1, col=1)
        self.metrics_fig.update_xaxes(title_text="Time", range=[0, self.T], row=2, col=1)
        self.metrics_fig.update_yaxes(range=[-0.05, 1.05], row=1, col=1)

        self.max_Rk = max_Rk
        self.max_wk = max_wk

    # ---------------------------------------------------------
    # Widgets
    # ---------------------------------------------------------
    def _init_widgets(self):
        self.play = widgets.Play(
            value=0, min=0, max=self.T-1,
            step=1, interval=50,
            description="Press play",
            show_repeat=False
        )

        self.slider = widgets.IntSlider(
            value=0, min=0, max=self.T-1,
            description="Time:",
            continuous_update=False,
            layout=widgets.Layout(width='300px')
        )

        widgets.jslink((self.play, 'value'), (self.slider, 'value'))
        self.slider.observe(self._update_frame, names='value')

        self.controls_box = widgets.HBox([self.play, self.slider])

    # ---------------------------------------------------------
    # Callback
    # ---------------------------------------------------------
    def _update_frame(self, change):
        t = int(change['new'])

        x_t = self.x_all[t]            # [K,N,3]
        ws_t = self.traj_w_np[t]       # [K,3]
        paths_t = self.traj_w_np[:t+1] # [<=T,K,3]

        # sphere indices
        idx_points0 = self._points_start_idx
        idx_w0 = self._w_start_idx
        idx_path0 = self._path_start_idx

        with self.sphere_fig.batch_update():
            # update fiber point clouds
            for k in range(self.K):
                xk = x_t[k]  # [N,3]
                tr = self.sphere_fig.data[idx_points0 + k]
                tr.x = xk[:, 0].tolist()
                tr.y = xk[:, 1].tolist()
                tr.z = xk[:, 2].tolist()

            # update w_k markers and paths
            for k in range(self.K):
                w_k = ws_t[k]
                self.sphere_fig.data[idx_w0 + k].x = [float(w_k[0])]
                self.sphere_fig.data[idx_w0 + k].y = [float(w_k[1])]
                self.sphere_fig.data[idx_w0 + k].z = [float(w_k[2])]

                path_k = paths_t[:, k, :]
                tr_path = self.sphere_fig.data[idx_path0 + k]
                tr_path.x = path_k[:, 0].tolist()
                tr_path.y = path_k[:, 1].tolist()
                tr_path.z = path_k[:, 2].tolist()

        # update metrics
        with self.metrics_fig.batch_update():
            # global R(t) point → data[1]
            self.metrics_fig.data[1].x = [t]
            self.metrics_fig.data[1].y = [float(self.global_order[t])]

            # max_k R_k(t) → data[3]
            self.metrics_fig.data[3].x = [t]
            self.metrics_fig.data[3].y = [float(self.max_Rk[t])]

            # max_k |w_k(t)| → data[4]
            self.metrics_fig.data[4].x = [t]
            self.metrics_fig.data[4].y = [float(self.max_wk[t])]

In [18]:

K   = 6          # number of fibers
N   = 64         # points per fiber
dim = 3          # S^2

# [K, N, 3] unit directions on S^2
fiber_points = torch.stack(
    [random_points_on_sphere(N, d=dim) for _ in range(K)],
    dim=0
)  # [K,N,3]

# Initial WS variables: for instance, all at hyperbolic origin
w0_all = torch.zeros(K, dim)

traj_w_all = integrate_fiber_full_cross_entropy_euler(
    w0_all=w0_all,
    fiber_points=fiber_points,
    dt=0.005,
    steps=300,
    dim=dim
)  # [T,K,3]
# 4) Visualization: push u_k^i through Möbius M_{w_k(t)} onto S^2
FiberCrossEntropyKuramoto3DWidget(traj_w_all, fiber_points)

    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'lightgrey', …